<a href="https://colab.research.google.com/github/davidekim/WRAPs/blob/main/miniCXCR4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Pipeline example for creating helical WRAPs for miniCXCR4**

In [ ]:
#@title **Get WRAPs git repo**
%%time
import os, time
import sys
import subprocess

def run_cmd(cmd):
  process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, shell=True, text=True)
  for line in iter(process.stdout.readline, ''):
    sys.stdout.write(line)
    sys.stdout.flush()
  process.stdout.close()
  process.wait()

if not os.path.isdir("WRAPs"):
  run_cmd("git clone https://github.com/davidekim/WRAPs.git")

In [ ]:
#@title setup **sushimaki** (~5-10min)
%%time
if not os.path.isdir("sushimaki"):
  print("installing sushimaki...")
  os.system("git clone https://github.com/davidekim/sushimaki.git")
  os.system("cd sushimaki; git submodule init; git submodule update --remote;")
  # install dependencies for ppi_iterative_opt submodule
  os.system("pip install jedi omegaconf hydra-core icecream pyrsistent pynvml decorator")
  os.system("pip install git+https://github.com/NVIDIA/dllogger#egg=dllogger")
  os.system("pip install --no-dependencies dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html")
  os.system("pip install --no-dependencies e3nn==0.5.5 opt_einsum_fx")
  os.system("cd sushimaki/ppi_iterative_opt/rf_diffusion/env/SE3Transformer; pip install .")
  os.system("pip install biopython==1.81")
  os.system("pip install -U dm-haiku")
  os.system("pip install ml-collections")
  os.system('pip install --upgrade "jax[cuda12_pip]<0.6.0" -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html')
  # install DeepTMHMM
  os.system("pip3 install --upgrade pybiolib")
  # install PyRosetta
  os.system("pip install pyrosetta --find-links https://west.rosettacommons.org/pyrosetta/quarterly/release")
  os.system("pip install py3Dmol")


In [ ]:
#@title Download RFdiffusion checkpoints (~1-5min)
%%time
if not os.path.exists("sushimaki/ppi_iterative_opt/rf_diffusion/models/BFF_4.pt"):
  os.system("mkdir sushimaki/ppi_iterative_opt/rf_diffusion/models")
  os.system("cd sushimaki/ppi_iterative_opt/rf_diffusion/models; wget https://files.ipd.uw.edu/pub/ppi_iterative_opt/rf_diffusion/models/BFF_4.pt")
if not os.path.exists("sushimaki/ppi_iterative_opt/rf_diffusion/models/base_complex_ss_finetuned_BFF_9.pt"):
  os.system("cd sushimaki/ppi_iterative_opt/rf_diffusion/models; wget http://files.ipd.uw.edu/pub/ppi_iterative_opt/rf_diffusion/models/base_complex_ss_finetuned_BFF_9.pt")

In [ ]:
#@title Download AF2 params (~5min)
%%time
if not os.path.isdir("sushimaki/ppi_iterative_opt/af2_initial_guess/params"):
  os.system("mkdir sushimaki/ppi_iterative_opt/af2_initial_guess/params")
  os.system("cd sushimaki/ppi_iterative_opt/af2_initial_guess/params; wget https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar; tar -xf alphafold_params_2022-12-06.tar")

In [ ]:
#@title **Create secondary structure and block adjacency inputs for C7 symmetric outputs**
%%time
#specify helix and loop length and connections
helix_length = 16
loop_length = 4
#specify number of helices and loops in ss_features input
no_helix = 3
no_loop = 2
#specify number of subunits
nsub = 7

features = f"1 H {helix_length}\n\
2 L {loop_length}\n\
3 H {helix_length}\n\
4 L {loop_length}\n\
5 H {helix_length}\n\
Intra 1,3 3,5\n\
Inter 5,1"

ss_folder = f'C{nsub}'
if not os.path.isdir(ss_folder):
  os.system(f"mkdir {ss_folder}")

calc_contig_length = (no_helix * helix_length) + (no_loop * loop_length)
print("calc_contig_length = ", calc_contig_length)
outname = ss_folder + '/ss_features.txt'
with open(outname, 'w') as f:
    f.writelines(features)
run_cmd(f'python WRAPs/helper_scripts/symm_make_ss_adj.py --def_file {outname} --nsub {nsub} --outfolder {ss_folder}')
from IPython.display import Image
Image(f'{ss_folder}/adj.png', width=600)

In [ ]:
#@title **Run pseudo-symmetric RFdiffusion** (~10min to a long time depending on num_designs)
%%time
num_designs = 5 #@param ["1","2","3","4","5","6","7","8","9","10"] {type:"raw"}
contig_length = calc_contig_length*nsub
cmd = f"python ./sushimaki/ppi_iterative_opt/rf_diffusion/run_inference.py --config-name symmetry "
cmd += f"\"contigmap.contigs=['{contig_length}-{contig_length}']\" "
cmd += f"inference.symmetry=pc{nsub} diffuser.T=50 scaffoldguided.scaffoldguided=True "
cmd += f"scaffoldguided.scaffold_dir=C{nsub} inference.output_prefix=miniCXCR4_symm_diffusion_wraps/miniCXCR4_symm_diffusion_wraps "
cmd += f"inference.num_designs={num_designs}"

print(cmd)
run_cmd(cmd)

import glob
pseudo_symm_wraps = []
for output in glob.glob('miniCXCR4_symm_diffusion_wraps/*.pdb'):
  pseudo_symm_wraps.append(output)

In [ ]:
#@title **Select wrap to use**
import py3Dmol
import ipywidgets as widgets
from ipywidgets import interact, Layout
from IPython.display import display, clear_output

current_wrap = ""
dropdown = widgets.Dropdown(
  options=pseudo_symm_wraps,
  description='wrap:',
  layout=Layout(width='50%', overflow='visible')
)
def on_dropdown_change(wrap):
  global current_wrap
  current_wrap = wrap
  clear_output(wait=True)
  view = py3Dmol.view(width=500, height=400)
  with open(wrap, "r") as f:
    pdb_data = f.read()
  view.addModel(pdb_data, 'pdb')
  view.setStyle({'cartoon': {'color':'magenta'}})
  view.zoomTo()
  view.show()

widgets.interact(on_dropdown_change, wrap=dropdown);

In [ ]:
#@title **Create miniCXCR4** (~5-10min)
%%time
import glob
from google.colab import files

pdb_code = '4rws'
contigs_str_4rws = "A23-45\,5-5\,A92-121\,5-5\,A164-206\,5-5\,A252-290"

num_designs = 2 #@param ["1","2","3","4","5"] {type:"raw"}

if not os.path.isfile(f"{pdb_code}.pdb1"):
    os.system(f"wget -qnc https://files.rcsb.org/download/{pdb_code}.pdb1.gz")
    os.system(f"gunzip {pdb_code}.pdb1.gz")

cmd = f"python ./sushimaki/ppi_iterative_opt/rf_diffusion/run_inference.py inference.output_prefix=miniCXCR4/miniCXCR4 "
cmd += f"inference.input_pdb={pdb_code}.pdb1 diffuser.T=30 "
cmd += f'contigmap.contigs=["{contigs_str_4rws}"] '
cmd += f"inference.num_designs={num_designs} denoiser.noise_scale_ca=0.5 denoiser.noise_scale_frame=0.5"
print(cmd)
run_cmd(cmd)

miniCXCR4s = []
for mini in glob.glob("miniCXCR4/miniCXCR4*.pdb"):
  miniCXCR4s.append(mini)

In [ ]:
#@title **Select miniCXCR4 to use**
import py3Dmol
import ipywidgets as widgets
from ipywidgets import interact, Layout
from IPython.display import display, clear_output

current_mini = ""
dropdown = widgets.Dropdown(
  options=miniCXCR4s,
  description='miniCXCR4:',
  layout=Layout(width='50%', overflow='visible')
)
def on_dropdown_change(mini):
  global current_mini
  current_mini = mini
  clear_output(wait=True)
  view = py3Dmol.view(width=500, height=400)
  with open(mini, "r") as f:
    pdb_data = f.read()
  view.addModel(pdb_data, 'pdb')
  view.setStyle({'cartoon': {'color':'magenta'}})
  view.zoomTo()
  view.show()

widgets.interact(on_dropdown_change, mini=dropdown);

In [ ]:
%%time
import glob
from google.colab import files

#@title **Place wrap around miniCXCR4 using sushimaki** (~1-5min)

# clear previous inputs
os.system("rm -rf input_DeepTMHMM input.pdb input_WRAP*.pdb partial_diffusion_task_file_input*.txt")
os.system(f"cp {current_mini} input.pdb")

# Since cxcr4 was truncated, DeepTMHMM cannot predict the transmembrane regions
# so we must specify which target positions represent the top, bottom, and all residues to wrap
# to guide the placement of the wrap
#
# Also, the top, bottom, and all positions of the wrap is automatically assigned using DSSP so
# if the wrap helical secondary structure is too broken up by loops or chain breaks from the
# pseudo-symmetric RFdiffusion step, the top, bottom, and all assignments may be off and will
# cause sushimaki to fail placing the wrap over the target. If this happens, you may want to
# make more pseudo-symmetric wraps and choose one with appropriate helical content.
cmd = f"python sushimaki/sushimaki.py "
cmd += f"--top_residues_to_wrap 12 37 44 72 97 124 136 "
cmd += f"--bottom_residues_to_wrap 22 28 55 63 106 112 150 "
cmd += f"--all_residues_to_wrap 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 "
cmd += f"44 45 46 47 48 49 50 51 52 53 54 55 56 57 58 59 60 61 62 63 64 65 66 67 68 69 70 71 72 "
cmd += f"97 98 99 100 101 102 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 "
cmd += f"136 137 138 139 140 141 142 143 144 145 146 147 148 149 150 "
cmd += f" --wrap {current_wrap} input.pdb"
print(cmd)
run_cmd(cmd)

target_wraps = []
for wrap in glob.glob('input_WRAP*.pdb'):
  target_wraps.append(wrap)

In [ ]:
#@title **Select wrapped target to use**
if len(target_wraps) == 0:
  raise Exception("sushimaki failed to place the wrap around the target. You may need to generate/select more compatible wraps")

current_wrapped_target = ""
dropdown = widgets.Dropdown(
  options=target_wraps,
  description='wrap:',
  layout=Layout(width='50%', overflow='visible')
)
def on_dropdown_change(wrap):
  global current_wrapped_target
  current_wrapped_target = wrap
  clear_output(wait=True)
  print()
  print(current_wrapped_target)
  view = py3Dmol.view(width=500, height=400)
  with open(wrap, "r") as f:
    pdb_data = f.read()
  view.addModel(pdb_data, 'pdb')
  view.setStyle({'cartoon': {'colorscheme': 'chain'}})
  view.zoomTo()
  view.show()

widgets.interact(on_dropdown_change, wrap=dropdown);



In [ ]:
#@title Run **ppi_iterative_opt** partial diffusion -> mpnn -> af2 optimization on selected wrap.
#@markdown Runtime depends on partial_T, partial_diffusions, total_traj, and cycles (~1 to many hours).

#@markdown Backbone diversity increases with partial_T. Hint: A value of 30 may help for larger or more difficult targets.
%%time
sushimaki_wrap = current_wrapped_target
partial_T = 30 #@param ["10", "15", "20", "25", "30"] {type:"raw"}
partial_diffusions = 5 #@param ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"] {type:"raw"}
total_traj = 1 #@param ["1", "2", "3", "4", "5"] {type:"raw"}
cycles = 1 #@param ["1", "2", "3", "4", "5", "6", "7", "8", "9", "10"] {type:"raw"}

# clear previous ppi_iterative_opt output
os.system("rm -rf ppi_iterative_opt_output; rm af2.sc; rm check.point_*;")

cmd = f"python sushimaki/ppi_iterative_opt/ppi_iterative_opt.py --partial_T {partial_T} --partial_diffusions {partial_diffusions} --cycles {cycles} --total_traj {total_traj} {sushimaki_wrap}"
print(cmd)
run_cmd(cmd)

In [ ]:
#@title **Plot AF2 plddt_binder vs pae_interaction of ppi_iterative_opt wraps**
import pandas as pd
import matplotlib.pyplot as plt
os.system("cat ppi_iterative_opt_output/*_af2.sc | head -n 1 > af2.sc")
os.system("cat ppi_iterative_opt_output/*_af2.sc | grep -v plddt_total | sort -n -k3 >> af2.sc")
df_af2 = pd.read_csv('af2.sc', sep=r'\s+')

def scatter_hist(x, y, ax, ax_histx, ax_histy, color, xlabel, ylabel):
    # no labels
    ax_histx.tick_params(axis="x", labelbottom=False)
    ax_histy.tick_params(axis="y", labelleft=False)

    # the scatter plot:
    ax.scatter(x, y)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax_histx.hist(x, bins=100)
    ax_histy.hist(y, orientation='horizontal', bins=100)

# Start with a square Figure.
fig = plt.figure(figsize=(8, 4))
gs = fig.add_gridspec(2, 2,  width_ratios=(4, 1), height_ratios=(1, 4),
                      left=0.1, right=0.9, bottom=0.1, top=0.9,
                      wspace=0.05, hspace=0.05)
# Create the Axes.
ax = fig.add_subplot(gs[1, 0])
ax_histx = fig.add_subplot(gs[0, 0], sharex=ax)
ax_histy = fig.add_subplot(gs[1, 1], sharey=ax)
# Draw the scatter plot and marginals.
scatter_hist(df_af2['pae_interaction'],df_af2['plddt_binder'], ax, ax_histx, ax_histy, 'black', 'pae interaction', 'plddt binder')


In [ ]:
#@title **Download AF2 WRAP**
af2_wraps = { 'Select to download': ''}
for i,r in df_af2.iterrows():
  name = r['description'].split('/')[-1]+'.pdb'
  af2_wraps[f"{name} pae_i: {r['pae_interaction']} plddt_binder: {r['plddt_binder']}"] = r['description']+'.pdb'

dropdown = widgets.Dropdown(
  options=af2_wraps,
  description='af2 wrap:',
  layout=Layout(width='50%', overflow='visible')
)
def on_dropdown_change(af2_wrap):
  if os.path.exists(af2_wrap):
    files.download(af2_wrap)

widgets.interact(on_dropdown_change, af2_wrap=dropdown);